# PART A — Baseline Notebook (OCTDL dataset)


## Cell 1 — Install/import libraries

In [1]:
!pip -q install xgboost lightgbm imbalanced-learn torch


In [2]:
import os
import time
import random
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms

from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, label_binarize, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)
from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from imblearn.over_sampling import SMOTE

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

print("All Library Import Successfully...")


All Library Import Successfully...


## Cell 2 — Reproducibility

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)


Device: cuda


## Cell 3a — Dataset Load

In [4]:
KAGGLE_INPUT_DIR = "/kaggle/input/datasets/orvile/octdl-optical-coherence-tomography-dataset"

print("Available datasets under /kaggle/input:")
for name in os.listdir(KAGGLE_INPUT_DIR):
    print(" -", name)

for root, dirs, files in os.walk(KAGGLE_INPUT_DIR):
    level = root.replace(KAGGLE_INPUT_DIR, "").count(os.sep)
    if level > 2:
        continue
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files[:5]:
        print(f"{indent}  {f}")


Available datasets under /kaggle/input:
 - OCTDL
octdl-optical-coherence-tomography-dataset/
  OCTDL/
    OCTDL_labels.csv
    OCTDL/


## Cell 3b — Load dataset + ResNet For Image Feature Extraction

In [5]:
LABELS_PATH = "/kaggle/input/datasets/orvile/octdl-optical-coherence-tomography-dataset/OCTDL/OCTDL_labels.csv"  
IMAGE_DIR = "/kaggle/input/datasets/orvile/octdl-optical-coherence-tomography-dataset/OCTDL/OCTDL"

assert os.path.exists(LABELS_PATH), f"labels csv : {LABELS_PATH}"
assert os.path.isdir(IMAGE_DIR), f"IMAGE_DIR : {IMAGE_DIR}"

labels_df = pd.read_csv("/kaggle/input/datasets/orvile/octdl-optical-coherence-tomography-dataset/OCTDL/OCTDL_labels.csv")
print("Labels shape:", labels_df.shape)
display(labels_df.head())

resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
resnet.fc = nn.Identity()
resnet.eval()
resnet.to(device)

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def find_image_path(file_name, image_dir):
    for ext in ["", ".jpg", ".jpeg", ".png", ".JPG", ".PNG"]:
        candidate = os.path.join(image_dir, file_name + ext)
        if os.path.exists(candidate):
            return candidate
    for root, _, files in os.walk(image_dir):
        for ext in ["", ".jpg", ".jpeg", ".png", ".JPG", ".PNG"]:
            target = file_name + ext
            if target in files:
                return os.path.join(root, target)
    return None

feature_rows = []
missing_files = []

with torch.no_grad():
    for fname in tqdm(labels_df["file_name"], desc="Extracting ResNet features"):
        img_path = find_image_path(str(fname), IMAGE_DIR)
        if img_path is None:
            missing_files.append(fname)
            feature_rows.append(np.full(2048, np.nan))
            continue

        img = Image.open(img_path).convert("RGB")
        tensor = preprocess(img).unsqueeze(0).to(device)
        feat = resnet(tensor).cpu().numpy().flatten()
        feature_rows.append(feat)

feature_cols = [f"feat_{i}" for i in range(2048)]
features_df = pd.DataFrame(feature_rows, columns=feature_cols)

df = pd.concat(
    [labels_df.reset_index(drop=True), features_df],
    axis=1
)

if missing_files:
    print(f"\nWARNING: {len(missing_files)} images not found, dropping those rows.")
    df = df.dropna(subset=feature_cols).reset_index(drop=True)

print("Final df shape:", df.shape)
display(df.head())


Labels shape: (2064, 10)


,file_name,disease,subcategory,condition,patient_id,eye,sex,year,image_width,image_hight
0,amd_1047099_1,AMD,intermediate,MNV_suspected,1047099,0,0,0,1101,410
1,amd_1047099_2,AMD,intermediate,MNV_suspected,1047099,0,0,0,731,265
2,amd_1047099_3,AMD,intermediate,MNV_suspected,1047099,0,0,0,1100,410
3,amd_1047099_4,AMD,intermediate,MNV_suspected,1047099,0,0,0,882,321
4,amd_1084498_1,AMD,late,MNV,1084498,0,0,0,882,321


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 221MB/s]
Extracting ResNet features: 100%|██████████| 2064/2064 [01:11<00:00, 28.82it/s]


Final df shape: (2064, 2058)


,file_name,disease,subcategory,condition,patient_id,eye,sex,year,image_width,image_hight,...,feat_2038,feat_2039,feat_2040,feat_2041,feat_2042,feat_2043,feat_2044,feat_2045,feat_2046,feat_2047
0,amd_1047099_1,AMD,intermediate,MNV_suspected,1047099,0,0,0,1101,410,...,0.019225,0.000000,0.000944,0.395437,0.004008,0.073663,0.035710,0.00193,0.145052,0.300740
1,amd_1047099_2,AMD,intermediate,MNV_suspected,1047099,0,0,0,731,265,...,0.000000,0.004709,0.069419,1.017439,0.000000,0.039197,1.756535,0.00000,0.030368,0.054500
2,amd_1047099_3,AMD,intermediate,MNV_suspected,1047099,0,0,0,1100,410,...,0.000000,0.000000,0.000000,0.091257,0.000000,0.021053,0.187012,0.00000,0.000000,0.000000
3,amd_1047099_4,AMD,intermediate,MNV_suspected,1047099,0,0,0,882,321,...,0.040185,0.000000,0.003222,0.898902,0.000000,0.130383,0.302829,0.00000,0.028156,0.044233
4,amd_1084498_1,AMD,late,MNV,1084498,0,0,0,882,321,...,0.000000,0.000000,0.000000,1.287179,0.000000,0.201777,0.194737,0.00000,0.000000,0.042913


## Cell 4 — Define target and group 


In [6]:
TARGET_COL = "disease"
GROUP_COL = "patient_id"

y = df[TARGET_COL].copy()
groups = df[GROUP_COL].copy()

non_feature_cols = [
    "file_name", "disease", "subcategory", "condition",
    "patient_id", "eye", "sex", "year",
    "image_width", "image_hight"
]
X = df.drop(columns=[c for c in non_feature_cols if c in df.columns])

print("X shape:", X.shape)
print("Classes:", y.nunique())
print("Class distribution:")
print(y.value_counts())


X shape: (2064, 2048)
Classes: 7
Class distribution:
disease
AMD    1231
NO      332
ERM     155
DME     147
RVO     101
VID      76
RAO      22
Name: count, dtype: int64


## Cell 5 — Patient/host-independent split

In [7]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

print("Train:", X_train.shape)
print("Test :", X_test.shape)

print(
    "Patient overlap:",
    len(set(groups_train).intersection(set(groups_test)))
)

Train: (1665, 2048)
Test : (399, 2048)
Patient overlap: 0


## Cell 6 — Validation split from training only

In [8]:
gss_val = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_idx2, val_idx = next(
    gss_val.split(
        X_train,
        y_train,
        groups=groups_train
    )
)

X_tr = X_train.iloc[train_idx2].copy()
X_val = X_train.iloc[val_idx].copy()

y_tr = y_train.iloc[train_idx2].copy()
y_val = y_train.iloc[val_idx].copy()

groups_tr = groups_train.iloc[train_idx2].copy()
groups_val = groups_train.iloc[val_idx].copy()

print("Train:", X_tr.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print(
    "Train-Val overlap:",
    len(set(groups_tr).intersection(set(groups_val)))
)


Train: (1349, 2048)
Validation: (316, 2048)
Test: (399, 2048)
Train-Val overlap: 0


## Cell 7 — Missing values handling

In [9]:
X_tr = X_tr.replace([np.inf, -np.inf], np.nan)
X_val = X_val.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

print("Missing values (train):", X_tr.isna().sum().sum())
print("Missing values (val)  :", X_val.isna().sum().sum())
print("Missing values (test) :", X_test.isna().sum().sum())

imputer = SimpleImputer(strategy="median")

X_tr_imp = pd.DataFrame(
    imputer.fit_transform(X_tr),
    columns=X_tr.columns,
    index=X_tr.index
)
X_val_imp = pd.DataFrame(
    imputer.transform(X_val),
    columns=X_val.columns,
    index=X_val.index
)
X_test_imp = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Missing-value handling completed.")
print("X_tr_imp shape:", X_tr_imp.shape)


Missing values (train): 0
Missing values (val)  : 0
Missing values (test) : 0
Missing-value handling completed.
X_tr_imp shape: (1349, 2048)


## Cell 8 — Scaling 

In [10]:
scaler = StandardScaler()

X_tr_scaled = scaler.fit_transform(X_tr_imp)
X_val_scaled = scaler.transform(X_val_imp)
X_test_scaled = scaler.transform(X_test_imp)

print("Scaling completed.")


Scaling completed.


## Cell 9 — Feature selection


In [11]:
K = min(100, X_tr_scaled.shape[1])

selector = SelectKBest(
    score_func=f_classif,
    k=K
)

X_tr_sel = selector.fit_transform(X_tr_scaled, y_tr)
X_val_sel = selector.transform(X_val_scaled)
X_test_sel = selector.transform(X_test_scaled)

print("Selected features:", X_tr_sel.shape[1])


Selected features: 100


## # Cell 10 — Handle imbalance data

In [12]:
smote = SMOTE(
    random_state=SEED
)

X_tr_bal, y_tr_bal = smote.fit_resample(
    X_tr_sel,
    y_tr
)

print("Before SMOTE:")
print(y_tr.value_counts())

print("\nAfter SMOTE:")
print(pd.Series(y_tr_bal).value_counts())

label_encoder = LabelEncoder()

y_tr_bal_enc = label_encoder.fit_transform(y_tr_bal)
y_test_enc = label_encoder.transform(y_test)

print("\nClasses:", list(label_encoder.classes_))
print("Encoded train label range:", y_tr_bal_enc.min(), "-", y_tr_bal_enc.max())

Before SMOTE:
disease
AMD    782
NO     228
ERM    100
DME     96
RVO     70
VID     56
RAO     17
Name: count, dtype: int64

After SMOTE:
disease
AMD    782
DME    782
ERM    782
NO     782
RAO    782
RVO    782
VID    782
Name: count, dtype: int64

Classes: ['AMD', 'DME', 'ERM', 'NO', 'RAO', 'RVO', 'VID']
Encoded train label range: 0 - 6


## Cell 11 — Prepare baseline models

In [13]:
models_dict = {

    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=SEED,
        n_jobs=-1
    ),

    "SVM": SVC(
        kernel="rbf",
        probability=True,
        random_state=SEED
    ),

    "k-NN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=SEED
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=SEED,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=SEED
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=SEED,
        n_jobs=-1
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        num_leaves=31,
        random_state=SEED,
        verbosity=-1
    ),

    "MLP": MLPClassifier(
        hidden_layer_sizes=(128, 64),
        max_iter=300,
        early_stopping=True,
        random_state=SEED
    )
}

## Cell 12 — Evaluation function

In [14]:
def evaluate_model(model, X_train, y_train, X_test, y_test, training_time):

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)
    else:
        y_prob = None

    result = {
        "Accuracy": accuracy_score(y_test, y_pred),

        "Precision_macro": precision_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "Recall_macro": recall_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "F1_macro": f1_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        ),

        "MCC": matthews_corrcoef(
            y_test,
            y_pred
        ),

        "Training_Time_sec": training_time
    }

    if y_prob is not None:
        try:
            result["ROC_AUC"] = roc_auc_score(
                y_test,
                y_prob,
                multi_class="ovr",
                average="macro"
            )

            y_test_bin = label_binarize(
                y_test,
                classes=np.unique(y_test)
            )

            result["PR_AUC"] = average_precision_score(
                y_test_bin,
                y_prob,
                average="macro"
            )

        except Exception:
            result["ROC_AUC"] = np.nan
            result["PR_AUC"] = np.nan

    else:
        result["ROC_AUC"] = np.nan
        result["PR_AUC"] = np.nan

    return result


## Cell 13 — Train all baselines

In [15]:
baseline_results = []
trained_models = {}

for name, model in models_dict.items():

    print(f"\nTraining: {name}")

    start_time = time.time()
    model.fit(X_tr_bal, y_tr_bal_enc)
    training_time = time.time() - start_time

    result = evaluate_model(
        model,
        X_tr_bal,
        y_tr_bal_enc,
        X_test_sel,
        y_test_enc,
        training_time
    )

    result["Model"] = name

    baseline_results.append(result)
    trained_models[name] = model

    print(result)


Training: Logistic Regression
{'Accuracy': 0.7092731829573935, 'Precision_macro': 0.48271420390606096, 'Recall_macro': 0.6101805825836274, 'F1_macro': 0.5264099385516527, 'MCC': np.float64(0.5611185170946148), 'Training_Time_sec': 1.7743158340454102, 'ROC_AUC': np.float64(0.9005390467225177), 'PR_AUC': np.float64(0.5460983907540637), 'Model': 'Logistic Regression'}

Training: SVM
{'Accuracy': 0.7844611528822055, 'Precision_macro': 0.6249016296194518, 'Recall_macro': 0.6075647522372459, 'F1_macro': 0.6048593929894298, 'MCC': np.float64(0.628356072640684), 'Training_Time_sec': 4.1033055782318115, 'ROC_AUC': np.float64(0.9151283064654155), 'PR_AUC': np.float64(0.6199353841737368), 'Model': 'SVM'}

Training: k-NN
{'Accuracy': 0.506265664160401, 'Precision_macro': 0.3878435758624446, 'Recall_macro': 0.5717499718648271, 'F1_macro': 0.41388582232267535, 'MCC': np.float64(0.4008808767550582), 'Training_Time_sec': 0.0008943080902099609, 'ROC_AUC': np.float64(0.7845971359510518), 'PR_AUC': np.f

## Cell 14 — Results table

In [16]:
baseline_df = pd.DataFrame(baseline_results)

baseline_df = baseline_df[
    [
        "Model",
        "Accuracy",
        "Precision_macro",
        "Recall_macro",
        "F1_macro",
        "ROC_AUC",
        "PR_AUC",
        "MCC",
        "Training_Time_sec"
    ]
].sort_values(
    "F1_macro",
    ascending=False
).reset_index(drop=True)

display(baseline_df)


,Model,Accuracy,Precision_macro,Recall_macro,F1_macro,ROC_AUC,PR_AUC,MCC,Training_Time_sec
0,SVM,0.784461,0.624902,0.607565,0.604859,0.915128,0.619935,0.628356,4.103306
1,XGBoost,0.774436,0.618677,0.604116,0.598095,0.880718,0.621688,0.613484,6.754932
2,Gradient Boosting,0.764411,0.598256,0.592293,0.586564,0.856698,0.594225,0.597888,180.456231
3,MLP,0.761905,0.579087,0.607910,0.579351,0.880349,0.586459,0.601579,1.152500
4,LightGBM,0.766917,0.655766,0.574185,0.577163,0.886135,0.619497,0.595752,4.758394
5,Random Forest,0.726817,0.698325,0.505975,0.548738,0.858815,0.574008,0.505907,4.132449
6,Logistic Regression,0.709273,0.482714,0.610181,0.526410,0.900539,0.546098,0.561119,1.774316
7,k-NN,0.506266,0.387844,0.571750,0.413886,0.784597,0.362916,0.400881,0.000894
8,Decision Tree,0.563910,0.340658,0.411299,0.363989,0.660559,0.232099,0.325393,0.917966


## Cell 15 — Find baseline to beat

In [17]:
best_baseline = baseline_df.iloc[0]

print("BASELINE TO BEAT")
print("----------------")
print("Model:", best_baseline["Model"])
print("Macro-F1:", best_baseline["F1_macro"])
print("ROC-AUC:", best_baseline["ROC_AUC"])
print("PR-AUC:", best_baseline["PR_AUC"])
print("MCC:", best_baseline["MCC"])


BASELINE TO BEAT
----------------
Model: SVM
Macro-F1: 0.6048593929894298
ROC-AUC: 0.9151283064654155
PR-AUC: 0.6199353841737368
MCC: 0.628356072640684
